# Representation geometry

PCA spectra and physical-state projections of frozen embeddings.
The analysis examines whether latent geometry reflects spatial coordinates
and periodic structure in the underlying environment.

## Embeddings

The [released data](../../README.md#pretrained-models) include seed-0 embeddings
for SMWM, SIGReg, and Forward-only. Select `METHOD` below. PCA is fitted on
training embeddings and used to project held-out examples.

The camera settings and PC order can be edited below.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "planning/train.py").is_file())
PLANNING = ROOT / "planning"
sys.path.insert(0, str(PLANNING))

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
from sklearn.decomposition import PCA
from analysis.probes import load_split, physical_targets
from plot_style import FULL_WIDTH, BLUE, RED, apply_matplotlib_style, position_colors

apply_matplotlib_style()
EXPORT_ROOT = PLANNING / "outputs/embeddings"
METHOD = "smwm"  # smwm, sigreg, forward_only

TWOROOM_PC_ORDER = (1, 2, 3)
TWOROOM_VIEW = (15, -143)  # elevation, azimuth
TWOROOM_ZOOM = 1.25
REACHER_VIEW = (22, -55)
CUBE_PC_ORDER = (1, 2)  # Use (1, 2, 3) for a 3D Cube panel.
CUBE_VIEW = (22, -55)
SCATTER_POINTS = 3000

ENVIRONMENTS = [
    ("tworoom", "TwoRoom", 2, ("Agent x", "Agent y")),
    ("pusht", "Push-T", 2, ("Agent x", "Agent y")),
    ("reacher", "Reacher", 2, ("Joint angle 1", "Joint angle 2")),
    ("ogbcube", "Cube", 5, ("Gripper x", "Gripper y")),
]

In [ ]:
def physical_coordinates(env, arrays):
    targets = physical_targets(env, arrays)
    if env == "reacher":
        key = "qpos" if "qpos" in arrays else "observation"
        return np.asarray(arrays[key]).reshape(len(arrays[key]), -1)[:, :2]
    quantity = "gripper position" if env == "ogbcube" else "agent position"
    return targets[quantity][:, :2]


def equal_limits(ax, points):
    center = (points.min(axis=0) + points.max(axis=0)) / 2
    radius = max(float(np.ptp(points, axis=0).max()) * 0.55, 1e-6)
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    if points.shape[1] == 3:
        ax.set_zlim(center[2] - radius, center[2] + radius)


def style_3d(ax, points, view, zoom=1.0):
    equal_limits(ax, points)
    ax.set_proj_type("ortho")
    ax.set_box_aspect((1, 1, 1), zoom=zoom)
    ax.view_init(elev=view[0], azim=view[1])
    ax.set(xticks=[], yticks=[], zticks=[])
    ax.grid(False)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((1, 1, 1, 0))
        axis.pane.set_edgecolor("0.85")
        axis.line.set_color("0.4")
        axis.line.set_linewidth(0.45)


representations = {}
for env, label, action_dim, state_labels in ENVIRONMENTS:
    directory = EXPORT_ROOT / env / METHOD
    if not all((directory / f"{split}_embeddings.pt").is_file() for split in ["train", "eval"]):
        print(f"No exported embeddings for {label}/{METHOD}.")
        continue
    z_train, _ = load_split(directory, "train")
    z_eval, arrays = load_split(directory, "eval")
    pca = PCA().fit(z_train)
    projected = pca.transform(z_eval)
    state = physical_coordinates(env, arrays)
    keep = np.isfinite(state).all(axis=1) & np.isfinite(projected).all(axis=1)
    if env == "pusht":
        low, high = np.nanpercentile(state, [0.5, 99.5], axis=0)
        keep &= ((state >= low) & (state <= high)).all(axis=1)
    indices = np.flatnonzero(keep)
    if len(indices) > SCATTER_POINTS:
        indices = np.random.default_rng(0).choice(indices, SCATTER_POINTS, replace=False)
    state = state[indices]
    representations[env] = (pca.explained_variance_ratio_, projected[indices], state, position_colors(state))

## Physical state and latent geometry

**Top:** PCA spectra, with action dimensions marked by red dashed lines.
**Middle:** distributions of representative physical coordinates.
**Bottom:** PC projections of the corresponding embeddings.

Physical states and embeddings share the same position colors. Reacher shows
the four three-dimensional projections of its first four PCs.

In [ ]:
if representations:
    fig = plt.figure(figsize=(FULL_WIDTH, 4.9))
    grid = fig.add_gridspec(
        3, 4, left=0.11, right=0.99, bottom=0.07, top=0.94,
        wspace=0.42, hspace=0.50,
    )
    for row, label in [(1, "Physical state"), (2, "Encoded embeddings")]:
        box = grid[row, 0].get_position(fig)
        fig.text(0.022, (box.y0 + box.y1) / 2, label,
                 rotation=90, ha="center", va="center", fontsize=10)

    for column, (env, label, action_dim, state_labels) in enumerate(ENVIRONMENTS):
        spectrum = fig.add_subplot(grid[0, column])
        physical = fig.add_subplot(grid[1, column])
        spectrum.set_title(label, pad=7)
        if env not in representations:
            geometry = fig.add_subplot(grid[2, column])
            for ax in [spectrum, physical, geometry]:
                ax.text(0.5, 0.5, "Embeddings unavailable", ha="center", va="center",
                        transform=ax.transAxes, fontsize=8)
                ax.set_axis_off()
            continue
        variance, projected, state, colors = representations[env]
        n = min(15, len(variance))
        spectrum.bar(np.arange(1, n + 1), variance[:n], width=0.55, color=BLUE)
        spectrum.axvline(action_dim + 0.5, color=RED, ls="--", lw=0.9,
                         label=f"action dim = {action_dim}")
        spectrum.set(xlim=(0.3, n + 0.7), xticks=[tick for tick in [1, 5, 10, 15] if tick <= n],
                     xlabel="Principal component")
        spectrum.set_ylim(0, max(0.05, variance[:n].max() * 1.1))
        spectrum.yaxis.set_major_locator(MaxNLocator(3))
        spectrum.legend(fontsize=6.5, loc="upper right", handlelength=1.2)
        spectrum.grid(axis="y")
        if column == 0:
            spectrum.set_ylabel("Explained variance ratio", fontsize=8)

        physical.scatter(*state.T, c=colors, s=1.8, linewidths=0, rasterized=True)
        equal_limits(physical, state)
        physical.set(xlabel=state_labels[0], ylabel=state_labels[1], xticks=[], yticks=[])
        physical.set_aspect("equal", adjustable="box")

        if env == "reacher":
            small_grid = grid[2, column].subgridspec(2, 2, wspace=0.12, hspace=0.22)
            combinations = [(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]
            for i, pcs in enumerate(combinations):
                ax = fig.add_subplot(small_grid[i // 2, i % 2], projection="3d")
                xyz = projected[:, pcs]
                ax.scatter(*xyz.T, c=colors, s=0.45, linewidths=0,
                           depthshade=False, rasterized=True)
                style_3d(ax, xyz, REACHER_VIEW, zoom=1.1)
                ax.text2D(0.5, 1.02, "PC" + "/".join(str(pc + 1) for pc in pcs),
                          ha="center", transform=ax.transAxes, fontsize=6)
                box = ax.get_position()
                ax.set_position([box.x0, box.y0 - 4 / (72 * fig.get_figheight()),
                                 box.width, box.height])
        elif env == "tworoom" or (env == "ogbcube" and len(CUBE_PC_ORDER) == 3):
            pcs = TWOROOM_PC_ORDER if env == "tworoom" else CUBE_PC_ORDER
            if sorted(pcs) != [1, 2, 3]:
                raise ValueError("The 3D PC order must be a permutation of (1, 2, 3).")
            xyz = projected[:, np.array(pcs) - 1]
            ax = fig.add_subplot(grid[2, column], projection="3d")
            ax.scatter(*xyz.T, c=colors, s=1.8, linewidths=0,
                       depthshade=False, rasterized=True)
            view = TWOROOM_VIEW if env == "tworoom" else CUBE_VIEW
            style_3d(ax, xyz, view, zoom=TWOROOM_ZOOM if env == "tworoom" else 1.1)
            ax.set_xlabel(f"PC{pcs[0]}", labelpad=-12)
            ax.set_ylabel(f"PC{pcs[1]}", labelpad=-12)
            ax.text2D(-0.11, 0.5, f"PC{pcs[2]}", rotation=90,
                      ha="center", va="center", transform=ax.transAxes)
        else:
            pcs = CUBE_PC_ORDER if env == "ogbcube" else (1, 2)
            xy = projected[:, np.array(pcs) - 1]
            ax = fig.add_subplot(grid[2, column])
            ax.scatter(*xy.T, c=colors, s=1.8, linewidths=0, rasterized=True)
            equal_limits(ax, xy)
            ax.set(xlabel=f"PC{pcs[0]}", ylabel=f"PC{pcs[1]}", xticks=[], yticks=[])
            ax.set_aspect("equal", adjustable="box")

    output = PLANNING / "outputs/analysis"
    output.mkdir(parents=True, exist_ok=True)
    fig.savefig(output / f"embeddings_{METHOD}.pdf")
    plt.show()
else:
    print("Download the analysis data or export a model's embeddings before plotting.")

The scatters use a fixed subsample. For Push-T, the outer 0.5% in each displayed
state coordinate is omitted to limit the effect of outliers; PCA uses the full
training embeddings. [Physical probes](physical_probes.ipynb) quantify the
recoverability of these quantities.